# Sohojatra AI — Fine-tuning Qwen2.5-3B-Instruct

**Platform**: Kaggle (T4 GPU × 2)  
**Model**: `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` + LoRA  
**Task**: Bangla civic concern urgency classification + emotion detection  
**Output**: LoRA adapter pushed to HuggingFace Hub (`sohojatra-lora`, private)

### Setup (one-time)
1. Upload `Sohojatra_Survey_Responses_FULL.csv` as a Kaggle Dataset named **sohojatra-survey**
2. Add your HuggingFace token as a Kaggle Secret named **HF_TOKEN** (Notebook settings → Secrets)
3. Enable GPU accelerator: T4 × 2

In [ ]:
%%capture
# Phase 1: Install dependencies
# Unsloth Kaggle edition — installs compatible torch/xformers automatically
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
    'unsloth', 'trl>=0.12.0', 'peft>=0.13.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.44.0',
    'datasets>=3.0.0', 'huggingface_hub>=0.26.0',
], check=True)
print('✅ Dependencies installed')

In [ ]:
# Phase 2: Configuration
import os
from huggingface_hub import login

# Read token from Kaggle secret (add it in Notebook settings → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found. Add it as a Kaggle secret or env var.')

login(token=HF_TOKEN, add_to_git_credential=False)

# ── Adjust these to your HuggingFace username ────────────────────────────────
HF_USERNAME = 'YOUR_HF_USERNAME'   # <-- replace with your HuggingFace username
HF_REPO_ID  = f'{HF_USERNAME}/sohojatra-lora'
OUTPUT_DIR  = '/kaggle/working/sohojatra-lora'
MAX_SEQ_LEN = 1024

print(f'Will push adapter to: {HF_REPO_ID}')

In [ ]:
# Phase 3: Build training dataset from survey CSV
import pandas as pd
import json
import re
import random
from collections import Counter
from datasets import Dataset

random.seed(42)

CSV_PATH = '/kaggle/input/sohojatra-survey/Sohojatra_Survey_Responses_FULL.csv'
df = pd.read_csv(CSV_PATH)
print(f'Survey rows: {len(df)}   Columns: {len(df.columns)}')

# ── Column finders ────────────────────────────────────────────────────────────
def col(partial):
    m = [c for c in df.columns if partial.lower() in c.lower()]
    return m[0] if m else None

CRITICAL_COL  = col('CRITICALLY URGENT')
MODERATE_COL  = col('MODERATE urgency')
LOW_COL       = col('LOW urgency')
EMOTION_COL   = col('emotions are commonly expressed')
LANG_CUE_COL  = col('LANGUAGE CUES signal urgency')
ANGER_COL     = col('ANGER in civic text')

print('Columns found:', [CRITICAL_COL[:30], MODERATE_COL[:30], LOW_COL[:30],
                          EMOTION_COL[:30], LANG_CUE_COL[:30], ANGER_COL[:30]])

def parse_multi(val):
    if pd.isna(val): return []
    return [v.strip() for v in str(val).split(',') if v.strip()]

def agg(col_name, top_n=15):
    if col_name is None: return []
    vals = []
    for row in df[col_name].dropna():
        vals.extend(parse_multi(row))
    return [v for v, _ in Counter(vals).most_common(top_n)]

def clean(text):
    """Strip Bangla parenthetical, e.g. 'Risk to life (জীবনের ঝুঁকি)' → 'Risk to life'"""
    return re.sub(r'\s*\([^)]*\)', '', str(text)).strip()

critical_factors = [clean(f) for f in agg(CRITICAL_COL)  if clean(f)]
moderate_factors = [clean(f) for f in agg(MODERATE_COL) if clean(f)]
low_factors      = [clean(f) for f in agg(LOW_COL)      if clean(f)]
emotions_raw     = agg(EMOTION_COL, top_n=10)
urgency_cues     = [clean(c) for c in agg(LANG_CUE_COL, top_n=8) if clean(c)]
anger_exprs      = [clean(a) for a in agg(ANGER_COL,    top_n=8) if clean(a)]

# ── Emotion normalisation ─────────────────────────────────────────────────────
EMOTION_NORM = [
    ('Frustration', 'frustration'), ('frustration', 'frustration'),
    ('Helplessness', 'helplessness'), ('despair', 'helplessness'),
    ('Fear', 'fear'), ('worry', 'fear'),
    ('Sarcasm', 'sarcasm'), ('humour', 'sarcasm'),
    ('Urgency', 'urgency'), ('panic', 'urgency'),
    ('Hope', 'hope'), ('optimism', 'hope'),
    ('Civic pride', 'civic_pride'), ('civic_pride', 'civic_pride'),
    ('Grief', 'grief'), ('sadness', 'grief'),
    ('Resignation', 'resignation'),
]

def norm_emotion(raw):
    for key, val in EMOTION_NORM:
        if key.lower() in raw.lower(): return val
    return None

emotions_normalized = list(dict.fromkeys(
    v for e in emotions_raw if (v := norm_emotion(e)) is not None
))
print('Critical factors:', critical_factors[:5])
print('Moderate factors:', moderate_factors[:5])
print('Emotions:', emotions_normalized)

In [ ]:
# Phase 3b: Generate synthetic ChatML training examples
SYSTEM = (
    'You are an AI assistant for the Sohojatra civic platform. '
    'You classify Bangla and Bilingual civic concern texts. '
    'Always reply with valid JSON only — no extra text.'
)

T_CRITICAL = [
    'আমাদের এলাকায় {f}। মানুষের জীবন বিপদে পড়েছে, এখনই জরুরি ব্যবস্থা নিন।',
    '{f} কারণে শিশু ও বৃদ্ধরা মারাত্মক ঝুঁকিতে আছেন। অবিলম্বে পদক্ষেপ চাই।',
    'জরুরি: {f} — ১০০ এর বেশি পরিবার সরাসরি ক্ষতিগ্রস্ত। দ্রুত হস্তক্ষেপ দরকার।',
    '{f} সমস্যায় হাসপাতালে যাওয়ার পথ বন্ধ। এখনই ব্যবস্থা না নিলে প্রাণহানি হবে।',
    'এখনই ব্যবস্থা নিন — {f}। মানুষের জীবনের ঝুঁকি রয়েছে।',
]
T_MODERATE = [
    'এলাকায় {f} সমস্যা প্রায় ৩ সপ্তাহ ধরে চলছে। কর্তৃপক্ষ দ্রুত সমাধান করুন।',
    '{f} নিয়ে পাড়ার মানুষ কষ্ট পাচ্ছেন। আর্থিক ক্ষতি হচ্ছে তবে জীবনহানির ঝুঁকি নেই।',
    'আমাদের ওয়ার্ডে {f} চলছে। সরকারি সেবা ব্যাহত হচ্ছে, সমাধান চাই।',
    '{f} কারণে ৫০-৬০ জন বাসিন্দা অসুবিধায় পড়ছেন। কর্তৃপক্ষের দৃষ্টি আকর্ষণ করছি।',
]
T_LOW = [
    'আমাদের এলাকায় {f} বিষয়ে একটি পরামর্শ দিতে চাই। জরুরি নয় তবে সমাধান হলে ভালো।',
    '{f} সমস্যাটি মৌসুমি এবং মাত্র ২-৩ জন প্রভাবিত। ধীরে সমাধান করা যেতে পারে।',
    'ছোট একটি বিষয়: {f}। এখন জরুরি নয়, তবে নজর দিলে ভালো হয়।',
]

def urgency_ex(text, level):
    user = f'নিচের নাগরিক অভিযোগ বিশ্লেষণ করুন এবং urgency level নির্ধারণ করুন (critical / moderate / low):\n\n"{text}"'
    return {'messages': [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': user},
        {'role': 'assistant', 'content': json.dumps({'urgency_level': level}, ensure_ascii=False)},
    ]}

def emotion_ex(text, emotions):
    user = f'নিচের নাগরিক পোস্টে কোন কোন আবেগ (emotion) প্রকাশ পাচ্ছে?\n\n"{text}"'
    return {'messages': [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': user},
        {'role': 'assistant', 'content': json.dumps({'emotions': emotions}, ensure_ascii=False)},
    ]}

examples = []

# Urgency examples
for factor in critical_factors[:10]:
    for tmpl in T_CRITICAL:
        examples.append(urgency_ex(tmpl.format(f=factor), 'critical'))

for factor in moderate_factors[:8]:
    for tmpl in T_MODERATE:
        examples.append(urgency_ex(tmpl.format(f=factor), 'moderate'))

for factor in low_factors[:6]:
    for tmpl in T_LOW:
        examples.append(urgency_ex(tmpl.format(f=factor), 'low'))

# Emotion examples (use urgency-cue phrases + critical texts)
neg_emotions = [e for e in ['frustration', 'fear', 'helplessness', 'urgency'] if e in emotions_normalized]
pos_emotions = [e for e in ['hope', 'civic_pride'] if e in emotions_normalized]

for i, factor in enumerate(critical_factors[:8]):
    text = T_CRITICAL[i % len(T_CRITICAL)].format(f=factor)
    emo  = random.sample(neg_emotions, min(3, len(neg_emotions))) if neg_emotions else ['urgency']
    examples.append(emotion_ex(text, emo))

for i, factor in enumerate(low_factors[:4]):
    text = T_LOW[i % len(T_LOW)].format(f=factor)
    emo  = random.sample(pos_emotions, min(2, len(pos_emotions))) if pos_emotions else ['hope']
    examples.append(emotion_ex(text, emo))

# Add urgency-cue examples (short one-liner texts that signal urgency)
for cue in urgency_cues[:6]:
    text = f'{cue} — আমাদের এলাকায় জরুরি অবস্থা তৈরি হয়েছে। দ্রুত সমাধান দরকার।'
    examples.append(urgency_ex(text, 'critical'))

random.shuffle(examples)
print(f'Total training examples: {len(examples)}')
print('Sample:', json.dumps(examples[0], ensure_ascii=False, indent=2)[:300])

dataset = Dataset.from_list(examples)
print(dataset)

In [ ]:
# Phase 4: Load base model + configure LoRA via Unsloth
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # auto-detect bf16/fp16
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,    # must be 0 for Unsloth fast CUDA kernels
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✅ Model + LoRA ready')
model.print_trainable_parameters()

In [ ]:
# Phase 5: Format dataset and train
from trl import SFTTrainer, SFTConfig

def formatting_func(examples):
    """
    Unsloth calls this two ways:
      - probe call : single example  → examples['messages'] = [{role,content}, ...]
      - train call : batch           → examples['messages'] = [[{role,content},...], ...]
    Must return a list[str] in both cases.
    """
    msgs = examples['messages']
    # Single example: first element is a dict  {role: ..., content: ...}
    # Batch:          first element is a list  [{role:...}, ...]
    if isinstance(msgs[0], dict):
        return [tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)]
    return [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in msgs]

# warmup_steps ≈ 10 % of total optimiser steps
total_steps = (len(dataset) // (4 * 4)) * 3   # (n // effective_batch) * epochs
warmup_steps = max(10, total_steps // 10)
print(f'Total steps ≈ {total_steps}   Warmup steps: {warmup_steps}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=warmup_steps,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy='epoch',
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        report_to='none',
        seed=42,
    ),
)

print('▶ Training...')
trainer_stats = trainer.train()
print(f'✅ Training done  —  steps: {trainer_stats.global_step}   '
      f'loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# Phase 6: Save locally and push LoRA adapter to HuggingFace Hub (private)
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ Saved locally to {OUTPUT_DIR}')

# Push adapter only (base model stays on HuggingFace)
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN, private=True)
print(f'✅ Adapter pushed → https://huggingface.co/{HF_REPO_ID}')
print()
print('Next step: set MODAL_API_URL after deploying ai/serve_model.py')

In [ ]:
# Phase 7: Quick inference sanity check
FastLanguageModel.for_inference(model)

def infer(text, task='urgency'):
    if task == 'urgency':
        user = f'নিচের নাগরিক অভিযোগ বিশ্লেষণ করুন এবং urgency level নির্ধারণ করুন (critical / moderate / low):\n\n"{text}"'
    else:
        user = f'নিচের নাগরিক পোস্টে কোন কোন আবেগ (emotion) প্রকাশ পাচ্ছে?\n\n"{text}"'

    # return_dict=True gives both input_ids and attention_mask — silences the pad/eos warning
    inputs = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to('cuda')

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=64,
            do_sample=False,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

tests = [
    ('আমাদের এলাকায় পানির লাইন ভেঙে গেছে, মানুষ পানি পাচ্ছে না। এখনই ব্যবস্থা নিন।', 'urgency'),
    ('রাস্তায় একটি ছোট গর্ত আছে। সুবিধামতো মেরামত করুন।', 'urgency'),
    ('কর্তৃপক্ষ কিছুই করছে না — এই দেশে কি আইন নেই? ক্লান্ত হয়ে গেছি।', 'emotion'),
]

for text, task in tests:
    result = infer(text, task)
    print(f'[{task}]  {text[:60]}...')
    print(f'  → {result}')
    print()